In [15]:
import asyncio
import random
from datetime import datetime
from collections import defaultdict
from obstore.store import S3Store

In [16]:
bucket = "s3://noaa-nexrad-level2/"
store = S3Store.from_url(bucket, skip_signature=True)

In [17]:
async def hourly_conus_breakdown(date_str: str) -> dict:
    """
    For a given date (YYYY/MM/DD), aggregates NEXRAD file count and size per hour (CONUS-wide).
    Returns: {hour: (count, size_in_bytes)}
    """
    hour_stats = defaultdict(lambda: [0, 0])  # hour -> [count, size]

    async for batch in store.list(date_str):
        for obj in batch:
            if not obj["last_modified"]:
                continue
            hour = obj["last_modified"].hour
            hour_stats[hour][0] += 1
            hour_stats[hour][1] += obj["size"]

    return hour_stats


def get_random_dates(n):
    dates = []
    for _ in range(n):
        year = random.randint(2014, 2024)
        month = random.randint(1, 12)
        day = random.randint(1, 28)  # safe default for all months
        dates.append(f"{year:04d}/{month:02d}/{day:02d}/")
    return dates

In [20]:
async def main():
    samples = 100  # increase this for more robust averaging
    dates = get_random_dates(samples)

    global_hour_stats = defaultdict(lambda: [0, 0])  # hour -> [count, total_size_bytes]

    print("\nSampling hourly data from:")
    for date in dates:
        print(f" - {date.rstrip('/')}")

        try:
            stats = await hourly_conus_breakdown(date)
            for hour in range(24):
                count, size = stats.get(hour, (0, 0))
                global_hour_stats[hour][0] += count
                global_hour_stats[hour][1] += size
        except Exception as e:
            print(f"  ⚠️ Failed on {date}: {e}")

    print(f"\n📊 Estimated Mean Hourly NEXRAD Stats (based on {samples} random days):")
    for hour in range(24):
        total_count, total_size = global_hour_stats[hour]
        mean_count = total_count / samples
        mean_gb = total_size / samples / (1024 ** 3)
        print(f"{hour:02d}:00 - {hour:02d}:59  |  {mean_count:6.1f} files/hr  |  {mean_gb:6.2f} GB/hr")

In [21]:
await main()


Sampling hourly data from:
 - 2015/09/20
 - 2023/03/11
 - 2022/02/24
 - 2023/10/20
 - 2023/02/01
 - 2014/06/13
 - 2015/12/26
 - 2022/05/25
 - 2019/12/11
 - 2017/07/25
 - 2024/03/09
 - 2015/07/17
 - 2018/09/12
 - 2018/08/11
 - 2020/08/14
 - 2017/05/10
 - 2019/05/19
 - 2023/11/26
 - 2023/08/18
 - 2016/09/16
 - 2015/05/01
 - 2018/03/01
 - 2017/06/21
 - 2019/06/15
 - 2022/09/14
 - 2021/11/04
 - 2016/02/15
 - 2014/11/26
 - 2016/03/08
 - 2016/01/15
 - 2020/07/04
 - 2017/02/15
 - 2014/12/07
 - 2014/03/25
 - 2021/03/20
 - 2014/04/26
 - 2024/04/09
 - 2019/11/24
 - 2015/01/02
 - 2022/02/17
 - 2023/10/01
 - 2021/04/11
 - 2022/08/24
 - 2016/05/03
 - 2023/06/04
 - 2015/08/07
 - 2014/04/14
 - 2014/04/18
 - 2014/01/06
 - 2023/10/13
 - 2016/10/06
 - 2024/02/25
 - 2018/08/13
 - 2023/01/05
 - 2020/10/05
 - 2021/03/28
 - 2015/02/14
 - 2014/10/24
 - 2019/04/01
 - 2016/02/08
 - 2021/02/09
 - 2020/11/04
 - 2020/07/17
 - 2018/01/17
 - 2016/10/21
 - 2019/07/13
 - 2014/08/15
 - 2024/06/28
 - 2017/09/21
 - 202